In [1]:
import pprint
import re

## Задание 1

In [2]:
PLATE_LETTERS = "АВЕКМНОРСТУХ"
LATIN_TO_CYRILLIC = str.maketrans("ABEKMHOPCTYX", PLATE_LETTERS)
CAR_ID_RE = re.compile(
    rf"^([{PLATE_LETTERS}]\d{{3}}[{PLATE_LETTERS}]{{2}})(\d{{2,3}})$"
)


def check_car_id(car_id: str) -> str:
    """Возвращает текст проверки номера. Латинские двойники приводятся к кириллице."""
    normalized = car_id.upper().translate(LATIN_TO_CYRILLIC)
    match = CAR_ID_RE.fullmatch(normalized)
    if match is None:
        return "Номер не валиден."

    number, region = match.groups()
    return f"Номер {number} валиден. Регион: {region}."


In [3]:
examples = {
    "А222ВС96": "Номер А222ВС валиден. Регион: 96.",
    "А123ВЕ777": "Номер А123ВЕ валиден. Регион: 777.",
    "АБ22ВВ193": "Номер не валиден.",
    "A222BC96": "Номер А222ВС валиден. Регион: 96.",
    "А222ВС9": "Номер не валиден.",
    "А222ВС9699": "Номер не валиден.",
    "Я222ВС96": "Номер не валиден.",
    "": "Номер не валиден.",
}

for car_id, expected in examples.items():
    result = check_car_id(car_id)
    print(f">>> check_car_id({car_id!r})")
    print(result)
    print()
    assert result == expected


>>> check_car_id('А222ВС96')
Номер А222ВС валиден. Регион: 96.

>>> check_car_id('А123ВЕ777')
Номер А123ВЕ валиден. Регион: 777.

>>> check_car_id('АБ22ВВ193')
Номер не валиден.

>>> check_car_id('A222BC96')
Номер А222ВС валиден. Регион: 96.

>>> check_car_id('А222ВС9')
Номер не валиден.

>>> check_car_id('А222ВС9699')
Номер не валиден.

>>> check_car_id('Я222ВС96')
Номер не валиден.

>>> check_car_id('')
Номер не валиден.



## Задание 2

In [4]:
LOG_LINE_RE = re.compile(
    r"^[ \t]*"
    r"(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) "
    r"(?P<level>[A-Z]+)"
    r"(?: (?P<body>.*?))?[ \t]*\r?$",
    re.MULTILINE,
)

FIELD_RE = re.compile(
    r"\b(?P<key>user|action|ip|card|msg)\s*=\s*"
    r"(?P<value>.*?)(?=\s+\b(?:user|action|ip|card|msg|extra|note|path|session|ua)\s*=|\s*$)"
)

EMAIL_RE = re.compile(
    r"^[A-Za-z0-9](?:[A-Za-z0-9_%+-]|\.[A-Za-z0-9_%+-])*"
    r"@[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?"
    r"(?:\.[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?)*"
    r"\.[A-Za-z]{2,}$"
)

IPV4_OCTET = r"(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)"
IPV4_RE = re.compile(rf"^{IPV4_OCTET}(?:\.{IPV4_OCTET}){{3}}$")
CARD_RE = re.compile(r"^\d{4}-\d{4}-\d{4}-(\d{4})$")
REPEAT_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+", flags=re.IGNORECASE)


def collapse_repeated_words(text: str) -> str:
    """«error error error timeout» → «error timeout»."""
    return REPEAT_RE.sub(r"\1", text)


def parse_logs(text: str) -> list[dict]:
    if not text or not text.strip():
        return []

    records = []
    for match in LOG_LINE_RE.finditer(text):
        body = match.group("body") or ""
        fields: dict[str, str] = {}
        for item in FIELD_RE.finditer(body):
            fields.setdefault(item.group("key"), item.group("value"))

        user = fields.get("user")
        action = fields.get("action")
        ip = fields.get("ip")
        raw_card = fields.get("card")

        errors: list[str] = []
        if user is None or EMAIL_RE.fullmatch(user) is None:
            errors.append("invalid_email")
        if ip is None or IPV4_RE.fullmatch(ip) is None:
            errors.append("invalid_ip")

        card = None
        if raw_card is not None:
            card_match = CARD_RE.fullmatch(raw_card)
            if card_match is None:
                errors.append("invalid_card")
            else:
                card = f"****-****-****-{card_match.group(1)}"

        record = {
            "ts": match.group("ts"),
            "level": match.group("level"),
            "user": user,
            "action": action,
            "ip": ip,
            "card": card,
        }
        if "msg" in fields:
            record["message"] = collapse_repeated_words(fields["msg"])
        record["errors"] = errors
        records.append(record)

    return records


In [5]:
sample = """\
2024-03-15 14:22:01 INFO user=ivan@mail.ru action=login ip=192.168.0.12
2024-03-15 14:22:05 WARN user=ivan@mail.ru action=retry ip=192.168.0.12
2024-03-15 14:23:11 ERROR user=bad..email@ action=pay ip=999.1.1.1 card=4111-1111-1111-1111
Mar 15 2024 14:24:00 INFO user=olga@company.com action=logout ip=10.0.0.5
"""

parsed = parse_logs(sample)
pprint.pp(parsed, width=88)

assert parsed == [
    {
        "ts": "2024-03-15 14:22:01",
        "level": "INFO",
        "user": "ivan@mail.ru",
        "action": "login",
        "ip": "192.168.0.12",
        "card": None,
        "errors": [],
    },
    {
        "ts": "2024-03-15 14:22:05",
        "level": "WARN",
        "user": "ivan@mail.ru",
        "action": "retry",
        "ip": "192.168.0.12",
        "card": None,
        "errors": [],
    },
    {
        "ts": "2024-03-15 14:23:11",
        "level": "ERROR",
        "user": "bad..email@",
        "action": "pay",
        "ip": "999.1.1.1",
        "card": "****-****-****-1111",
        "errors": ["invalid_email", "invalid_ip"],
    },
]


[{'ts': '2024-03-15 14:22:01',
  'level': 'INFO',
  'user': 'ivan@mail.ru',
  'action': 'login',
  'ip': '192.168.0.12',
  'card': None,
  'errors': []},
 {'ts': '2024-03-15 14:22:05',
  'level': 'WARN',
  'user': 'ivan@mail.ru',
  'action': 'retry',
  'ip': '192.168.0.12',
  'card': None,
  'errors': []},
 {'ts': '2024-03-15 14:23:11',
  'level': 'ERROR',
  'user': 'bad..email@',
  'action': 'pay',
  'ip': '999.1.1.1',
  'card': '****-****-****-1111',
  'errors': ['invalid_email', 'invalid_ip']}]


In [6]:
assert parse_logs("") == []
assert parse_logs("   \n\t") == []

assert collapse_repeated_words("error error error timeout") == "error timeout"
assert collapse_repeated_words("функцию функцию, которая") == "функцию, которая"

boundary = """\
2024-03-15 09:39:44 WARN user=user+tag@gmail.com action=retry ip=256.0.0.1
2024-03-15 09:27:50 WARN user=a.b@domain.co action=retry ip=1.2.3
2024-03-15 09:53:37 ERROR user=a.b@domain.co action=pay ip=8.8.4.4 card=4111111111111111
2024-03-15 11:52:20 DEBUG user=test_user@sub.domain.org action=export ip=10.0.0.5 msg=error error error timeout
2024-03-15 10:56:31 WARN
2024-03-15 11:24:00 INFO user = olga@company.com action = login ip = 8.8.4.4
"""

by_ip = {record["ip"]: record for record in parse_logs(boundary)}

invalid_octet = by_ip["256.0.0.1"]
assert invalid_octet["errors"] == ["invalid_ip"]
assert invalid_octet["user"] == "user+tag@gmail.com"

short_ip = by_ip["1.2.3"]
assert short_ip["errors"] == ["invalid_ip"]

raw_card = next(record for record in parse_logs(boundary) if record["action"] == "pay")
assert raw_card["card"] is None
assert raw_card["errors"] == ["invalid_card"]

with_message = next(record for record in parse_logs(boundary) if record["level"] == "DEBUG")
assert with_message["message"] == "error timeout"
assert "message" not in invalid_octet

empty_tail = next(record for record in parse_logs(boundary) if record["action"] is None)
assert empty_tail["user"] is None
assert empty_tail["ip"] is None
assert empty_tail["errors"] == ["invalid_email", "invalid_ip"]

spaced = next(record for record in parse_logs(boundary) if record["user"] == "olga@company.com")
assert spaced["errors"] == []
assert spaced["action"] == "login"
assert spaced["ip"] == "8.8.4.4"

print("пограничные случаи: ок")


пограничные случаи: ок


In [7]:
try:
    with open("logs_hw4.txt", encoding="utf-8") as log_file:
        logs = parse_logs(log_file.read())
except FileNotFoundError:
    logs = None
    print("Файл logs_hw4.txt не найден")

if logs is not None:
    with_errors = sum(bool(record["errors"]) for record in logs)
    with_message = sum("message" in record for record in logs)
    print(f"Разобрано записей: {len(logs)}")
    print(f"С ошибками: {with_errors}")
    print(f"С сообщением: {with_message}")
    sample_message = next(
        record for record in logs if record.get("message") == "error timeout"
    )
    pprint.pp(sample_message, width=88)


Разобрано записей: 593
С ошибками: 295
С сообщением: 43
{'ts': '2024-03-15 11:52:20',
 'level': 'DEBUG',
 'user': 'test_user@sub.domain.org',
 'action': 'export',
 'ip': '10.0.0.5',
 'card': None,
 'message': 'error timeout',
 'errors': []}
